# 03b - Limpieza y modelo básico
**Maestría en IA - UDIT** · Lenguajes de Programación para IA · Prof. Carlos Cob
Alumno: Santiago Toledo Berral (STB)

Primer modelo de precio: una regresión Ridge sobre el nombre del producto (TF-IDF) más algunas variables numéricas. Es sencillo y se puede interpretar: cada palabra tiene un peso que sube o baja el precio.

La limpieza está en `src/artifacts/limpieza.py`, para que el modelo guardado aplique las mismas reglas a productos nuevos.

In [1]:
import json
from datetime import date
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Ridge
from sklearn.metrics import make_scorer, mean_absolute_error, r2_score
from sklearn.model_selection import GroupKFold, GroupShuffleSplit, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler

import sys
sys.path.insert(0, "../src")
from artifacts import limpieza as stb

pio.renderers.default = "plotly_mimetype+notebook_connected"
pio.templates.default = "plotly_white"
AZUL, ROJO, GRIS = "#1c5cab", "#c23a3a", "#898781"
SEMILLA = 42
DATA_DIR, MODELS_DIR = Path("../data"), Path("../models")
MODELS_DIR.mkdir(exist_ok=True)

## 1. Limpieza
Mismas reglas que salieron del EDA (notebook 02): quitar filas sin texto o con precio <= 0 y duplicados exactos, separar los campos del texto, normalizar unidades y marcar (sin borrar) las cantidades inválidas. Los outliers de precio no se eliminan.

In [2]:
df_raw = pd.read_csv(DATA_DIR / "train_full.csv")
df, reporte = stb.limpiar_dataset(df_raw)
display(reporte)
df[["item_name", "value", "unit_raw", "unit_family", "qty_std", "pack_count", "price"]].head()

,paso,filas,detalle
0,0. Dataset original,75000,-
1,1. Eliminar filas sin texto o con precio <= 0,75000,0 eliminadas
2,2. Eliminar duplicados exactos (texto + precio),74973,27 eliminadas
3,"3. Parsear nombre, bullets, descripción, Value...",74973,5 campos nuevos
4,4. Normalizar unidades → Peso / Volumen / Conteo,74973,64 grafías → 4 familias
5,"5. Marcar cantidades inválidas (vacías, 0 o > ...",74973,999 marcadas (no se borran)
6,6. Extraer packs y variables de texto,74973,"pack_count, has_description, n_words"
7,7. Crear log_price e image_group,74973,target y grupos para validación


,item_name,value,unit_raw,unit_family,qty_std,pack_count,price
0,"La Victoria Green Taco Sauce Mild, 12 Ounce (P...",72.00,Fl Oz,Volumen,72.00,6,4.89
1,"Salerno Cookies, The Original Butter Cookies, ...",32.00,Ounce,Peso,32.00,4,13.12
2,"Bear Creek Hearty Soup Bowl, Creamy Chicken wi...",11.40,Ounce,Peso,11.40,6,1.97
3,Judee’s Blue Cheese Powder 11.25 oz - Gluten-F...,11.25,Ounce,Peso,11.25,1,30.34
4,"kedem Sherry Cooking Wine, 12.7 Ounce - 12 per...",12.00,Count,Conteo,12.00,12,66.49


## 2. Variables y separación
`stb.construir_features()` convierte el texto crudo en: el nombre (que pasa por TF-IDF), log de la cantidad, familia de unidad, log del pack, nº de bullets, si tiene descripción y nº de palabras.

Separo train y test (80/20) agrupando por imagen, para que las variantes de un mismo producto no queden repartidas entre los dos.

In [3]:
display(stb.construir_features(df_raw.head(3)))

X, y, grupos = df[["catalog_content"]], df["log_price"], df["image_group"]
idx_train, idx_test = next(GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEMILLA).split(X, y, grupos))
X_train, X_test, y_train, y_test = X.iloc[idx_train], X.iloc[idx_test], y.iloc[idx_train], y.iloc[idx_test]
precio_test = df["price"].iloc[idx_test].to_numpy()
print(f"Train: {len(X_train):,} | Test: {len(X_test):,}")

,nombre,log_qty,qty_missing,unit_family,log_pack,n_bullets,has_description,log_words
0,"la victoria green taco sauce mild, 12 ounce (p...",4.290459,0,Volumen,1.791759,0,0,2.944439
1,"salerno cookies, the original butter cookies, ...",3.496508,0,Peso,1.386294,5,0,4.394449
2,"bear creek hearty soup bowl, creamy chicken wi...",2.517696,0,Peso,1.791759,5,0,4.094345


Train: 59,964 | Test: 15,009


## 3. Comparación de modelos
1. Predecir siempre la mediana (referencia).
2. Ridge solo con variables numéricas.
3. Ridge con numéricas + TF-IDF del nombre.

Las métricas se calculan en dólares, después de deshacer el log: SMAPE (la métrica del challenge), MAE y R² sobre el log.

In [4]:
def crear_modelo(alpha=1.0, usar_texto=True):
    pasos = [("num", StandardScaler(), stb.COLUMNAS_NUMERICAS),
             ("cat", OneHotEncoder(handle_unknown="ignore"), stb.COLUMNAS_CATEGORICAS)]
    if usar_texto:
        pasos.append(("tfidf", TfidfVectorizer(ngram_range=(1, 2), min_df=3, sublinear_tf=True), "nombre"))
    return Pipeline([("features", FunctionTransformer(stb.construir_features)),
                     ("prep", ColumnTransformer(pasos)),
                     ("modelo", Ridge(alpha=alpha))])

def evaluar(nombre, pred_log):
    pred = np.expm1(pred_log)
    return {"modelo": nombre, "SMAPE (%)": stb.smape(precio_test, pred),
            "MAE ($)": mean_absolute_error(precio_test, pred), "R² (log)": r2_score(y_test, pred_log)}

tabla = pd.DataFrame([
    evaluar("Mediana", np.full(len(y_test), y_train.median())),
    evaluar("Ridge solo numéricas", crear_modelo(usar_texto=False).fit(X_train, y_train).predict(X_test)),
    evaluar("Ridge numéricas + nombre", crear_modelo().fit(X_train, y_train).predict(X_test)),
]).set_index("modelo")
tabla.round(3)

,SMAPE (%),MAE ($),R² (log)
modelo,,,
Mediana,73.344,16.845,-0.002
Ridge solo numéricas,67.625,15.667,0.144
Ridge numéricas + nombre,51.545,11.903,0.493


Las variables numéricas solas mejoran poco a la mediana. El salto lo da el nombre del producto: importa más qué es que cuánto trae.

## 4. Ajuste de `alpha`
`alpha` es la regularización del Ridge. Lo elijo con validación cruzada de 5 folds agrupada por imagen, solo sobre train.

In [5]:
scorer = make_scorer(lambda yt, yp: stb.smape(np.expm1(yt), np.expm1(yp)), greater_is_better=False)
cv_res = {}
for a in [0.3, 0.6, 1.0, 2.0, 4.0]:
    s = -cross_val_score(crear_modelo(alpha=a), X_train, y_train, groups=grupos.iloc[idx_train],
                         cv=GroupKFold(n_splits=5), scoring=scorer)
    cv_res[a] = s.mean()
    print(f"alpha={a:<4} SMAPE CV = {s.mean():.2f} ± {s.std():.2f}")
MEJOR_ALPHA = min(cv_res, key=cv_res.get)
print("Mejor alpha:", MEJOR_ALPHA)

alpha=0.3  SMAPE CV = 53.96 ± 0.54


alpha=0.6  SMAPE CV = 52.71 ± 0.51


alpha=1.0  SMAPE CV = 52.33 ± 0.48


alpha=2.0  SMAPE CV = 52.54 ± 0.43


alpha=4.0  SMAPE CV = 53.47 ± 0.42
Mejor alpha: 1.0


## 5. Evaluación en test

In [6]:
modelo = crear_modelo(alpha=MEJOR_ALPHA).fit(X_train, y_train)
pred_log = modelo.predict(X_test)
pred = np.expm1(pred_log)
metricas_test = evaluar(f"Ridge + nombre (alpha={MEJOR_ALPHA})", pred_log)
print({k: round(v, 3) if isinstance(v, float) else v for k, v in metricas_test.items()})

tramo = pd.cut(precio_test, [0, 5, 10, 20, 50, 100, np.inf], labels=["< $5", "$5-10", "$10-20", "$20-50", "$50-100", "> $100"])
pd.DataFrame({"tramo": tramo, "real": precio_test, "pred": pred}).groupby("tramo", observed=True).apply(
    lambda g: pd.Series({"SMAPE (%)": stb.smape(g["real"], g["pred"]), "productos": len(g),
                         "sesgo mediano (%)": np.median((g["pred"] - g["real"]) / g["real"]) * 100}),
    include_groups=False).round(1)

{'modelo': 'Ridge + nombre (alpha=1.0)', 'SMAPE (%)': 51.545, 'MAE ($)': 11.903, 'R² (log)': 0.493}


,SMAPE (%),productos,sesgo mediano (%)
tramo,,,
< $5,76.9,2670.0,116.8
$5-10,43.5,3091.0,36.3
$10-20,34.5,3732.0,0.0
$20-50,47.3,3878.0,-28.0
$50-100,67.9,1254.0,-47.6
> $100,94.4,384.0,-62.9


El modelo acierta mejor en el rango central y tira hacia el centro: sobreestima los productos muy baratos y subestima los muy caros.

## 6. Qué aprendió el modelo
Al ser lineal se pueden leer los coeficientes de cada palabra: positivo sube el precio, negativo lo baja.

In [7]:
coefs = pd.Series(modelo.named_steps["modelo"].coef_, index=modelo.named_steps["prep"].get_feature_names_out())
palabras = coefs[coefs.index.str.startswith("tfidf__")].rename(lambda s: s.replace("tfidf__", ""))
extremos = pd.concat([palabras.nsmallest(12), palabras.nlargest(12)]).sort_values()

fig = go.Figure(go.Bar(y=extremos.index, x=extremos.values, orientation="h",
                       marker_color=np.where(extremos.values > 0, ROJO, AZUL)))
fig.update_layout(title="Palabras que más suben (rojo) o bajan (azul) el precio", xaxis_title="Coeficiente en log(precio)", height=650)
fig.show()

Suben el precio las palabras de formato grande o mayorista (*pounds, case, gallon*) y de regalo (*gift*). Lo bajan las marcas económicas de supermercado (*great value, campbell, maruchan*).

## 7. Guardar el modelo
Reentreno con todos los datos y guardo el pipeline completo (del texto crudo al log del precio).

In [8]:
modelo_final = crear_modelo(alpha=MEJOR_ALPHA).fit(X, y)
joblib.dump(modelo_final, MODELS_DIR / "stb_ridge_tfidf_v1.joblib")
meta = {"fecha": date.today().isoformat(), "alpha": MEJOR_ALPHA, "filas": len(X),
        "metricas_test": {k: round(v, 4) for k, v in metricas_test.items() if k != "modelo"}}
(MODELS_DIR / "stb_ridge_tfidf_v1.json").write_text(json.dumps(meta, indent=2, ensure_ascii=False), encoding="utf-8")

# Prueba con productos nuevos
cargado = joblib.load(MODELS_DIR / "stb_ridge_tfidf_v1.joblib")
nuevos = pd.DataFrame({"catalog_content": [
    "Item Name: Organic Extra Virgin Olive Oil, 16.9 Fl Oz Glass Bottle\nValue: 16.9\nUnit: Fl Oz\n",
    "Item Name: Organic Extra Virgin Olive Oil, 16.9 Fl Oz (Pack of 6)\nValue: 101.4\nUnit: Fl Oz\n",
    "Item Name: Campbell's Condensed Chicken Noodle Soup, 10.75 Ounce Can\nValue: 10.75\nUnit: Ounce\n"]})
np.expm1(cargado.predict(nuevos)).round(2)

array([14.46, 26.35,  1.65])

## Conclusiones
- El Ridge con el nombre del producto baja el SMAPE de ~73% (mediana) a ~51,5% en test. La validación cruzada da un valor parecido, así que no está sobreajustado.
- La señal principal es el nombre del producto; cantidad y pack ayudan poco solos.
- Limitaciones: trata la cantidad igual para todos los productos (no capta que un pack de 6 aceites cueste ~6 veces más), tira hacia el centro y no usa la imagen.
- En el notebook 04 se mejora con clusters de producto, precios típicos por marca y LightGBM.